In [1]:
#import csv
import json
import requests
import pandas as pd
from tqdm import tqdm  # progress_apply
import postgis as pg
from pymeos import *
import pickle
# from tqdm.notebook import tqdm
import asyncio
import aiohttp
import os
from shapely.geometry import Point


# -----------------------
# CONFIGURATION
# -----------------------
SERVER_URL = "http://localhost:8080"
COLLECTION_ID = "ships"           # Change to your collection name


LIMIT_ROWS = None
BATCH_SIZE = 50


/home/sirine/Desktop/MobilityAPI/venv/lib/python3.13/site-packages/postgis/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### based on PYMEOS example : https://github.com/MobilityDB/PyMEOS-Examples/blob/main/PyMEOS_Examples/AIS.ipynb
### -----------------------
### HELPER FUNCTIONS
### -----------------------

### rework the data using pandas and pymeos to get temporal geometry (trip)

In [2]:
tqdm.pandas()

pymeos_initialize()
# %%time
# get data, cleanup remove duplicates, wrong lat,long ///

ais = pd.read_csv(
    "./data/aisdk-2026-03-25.zip",
    usecols=["# Timestamp", "MMSI", "Latitude", "Longitude", "SOG","COG", "Heading","Name","Ship type","Destination","IMO"],
    nrows=2000
)
ais.columns = ["t", "mmsi", "lat", "lon", "SOG","COG","Heading","Name","Ship type","Destination","IMO"]
# data preparation:  remove rows with no timestamps
ais = ais[ais["t"] != 0]
ais["t"] = pd.to_datetime(ais["t"], format='%d/%m/%Y %H:%M:%S')
ais = ais[ais["mmsi"] != 0]
ais = ais.drop_duplicates(["t", "mmsi"])
ais = ais[(ais["lat"] >= 40.18) & (ais["lat"] <= 84.17)]
ais = ais[(ais["lon"] >= -16.1) & (ais["lon"] <= 32.88)]
ais = ais[(ais["SOG"] >= 0) & (ais["SOG"] <= 1022)]
ais["Name"] = ais["Name"].apply(lambda x: "No Name" if x == '' else x)
ais["COG"] = pd.to_numeric(ais["COG"], errors='coerce')
ais["SOG"] = pd.to_numeric(ais["SOG"], errors='coerce')
ais["Heading"] = pd.to_numeric(ais["Heading"], errors='coerce')
ais = ais[ais["COG"].notna()]
ais = ais[ais["SOG"].notna()]
ais = ais[ais["Heading"].notna()]
ais = ais[ais["Destination"] != "Undefined"]
temp  = ais["Destination"]
ais["Name"] = ais["Ship type"]
ais["Ship type"] = temp
temp = ais["IMO"]
ais["IMO"] = ais["Name"]
ais["Destination"]= temp
ais.dropna()
ais.head()

,t,mmsi,lat,lon,SOG,COG,Heading,Name,Ship type,Destination,IMO
584,2026-03-25 00:00:02,219016577,54.599802,11.383923,3.2,22.7,17.0,BUILDER GUARD VESSEL,HSC,GUARD VESSEL BUILDER,BUILDER GUARD VESSEL
633,2026-03-25 00:00:03,219679000,55.462687,8.444885,0.0,16.6,50.0,SIGYN,Tug,ESBJERG,SIGYN
752,2026-03-25 00:00:03,219002565,55.268220,9.888073,0.0,247.3,288.0,BARSOFARGEN,Passenger,BARSOE - LANDINGEN,BARSOFARGEN
939,2026-03-25 00:00:04,273257370,57.498545,8.300530,11.4,236.4,236.0,IVAN AIVAZOVSKY,Tanker,TR ALI,IVAN AIVAZOVSKY
940,2026-03-25 00:00:04,219358000,57.859537,10.080503,17.6,247.8,248.0,FLANDRIA SEAWAYS,Cargo,GOT-ZEE-GOT,FLANDRIA SEAWAYS


# Now, we will create the PyMEOS object representing
# the position and the SOG.

In [3]:

ais["inst"] = ais.progress_apply(
    lambda row: TGeomPointInst.from_base_time(Point(row["lon"], row["lat"]), row["t"]),
    axis=1
)

ais["sog_inst"] = ais.progress_apply(
    lambda row: TFloatInst(value=row["SOG"], timestamp=row["t"]),
    axis=1
)

ais["cog_inst"] = ais.progress_apply(
    lambda row: TFloatInst(value=row["COG"], timestamp=row["t"]),
    axis=1
)
ais["heading_inst"] = ais.progress_apply(
    lambda row: TFloatInst(value=row["Heading"], timestamp=row["t"]),
    axis=1
)

ais.drop(["lat", "lon"], axis=1, inplace=True)

tel = ais.groupby("mmsi").size()
mel = ais.groupby("mmsi")["t"].nunique()
# print(tel, mel)
trajectories = ais.groupby("mmsi").aggregate(
    {
        "inst": lambda x: TGeomPointSeq.from_instants(list(x), upper_inc=True),
        "sog_inst": lambda x: TFloatSeq.from_instants(list(x), upper_inc=True),
        "cog_inst": lambda x: TFloatSeq.from_instants(list(x), upper_inc=True),
        "heading_inst": lambda x: TFloatSeq.from_instants(list(x), upper_inc=True),
        "Name": "first",
        "IMO": "first",
        "Ship type":"first",
        "Destination": "first"
    }
).rename({"inst": "trajectory", "sog_inst": "SOG","cog_inst":"COG","heading_inst":"Heading"}, axis=1)


json_ready = []
for mmsi, row in trajectories.iterrows():
    # print(type(row["trajectory"]))
    json_ready.append({
        "mmsi": int(mmsi),
        "trajectory": json.loads(row["trajectory"].as_mfjson()),
        "SOG": json.loads(row["SOG"].as_mfjson()),
        "COG": json.loads(row["COG"].as_mfjson()),
        "Heading": json.loads(row["Heading"].as_mfjson()),
        "properties":{
            "Name": row["Name"],
            "IMO": row["IMO"],
            "Ship type":row["Ship type"],
            "Destination": row["Destination"]
        }
       
    })


with open("data/trajectories_mf1.json", "w") as f:
    json.dump(json_ready, f, indent=2)
    





100%|██████████| 33/33 [00:00<00:00, 26710.16it/s]


# Assembling Trips (MEOS Example)
# Now, we will create the trajectory (TGeogPointSeq) and
# the SOG evolution (TFloatSeq) for every ship (identified by the mmsi)
# using the instants we have created.
# ::::::::::::::::::::::::::::stopped here , time limit

In [4]:
pymeos_finalize() 
# trajectories = (
#     ais.groupby("mmsi")
#     .aggregate(
#         {
#             "instant": lambda x: TGeogPointSeq.from_instants(x, upper_inc=True),
#             "sog": lambda x: TFloatSeq.from_instants(x, upper_inc=True),
#         }
#     )
#     .rename({"instant": "trajectory"}, axis=1)
# )
# trajectories["distance"] = trajectories["trajectory"].apply(
#     lambda t: t.length())

# trajectories.head()


In [5]:


# # Convert PyMEOS objects to MFJSON
# trajectories_json = trajectories.reset_index()  # mmsi becomes a column
# trajectories_json["trajectory"] = trajectories_json["trajectory"].apply(
#     lambda t: json.loads(t.as_mfjson())
# )
# trajectories_json["sog"] = trajectories_json["sog"].apply(
#     lambda t: json.loads(t.as_mfjson())
# )
# # 

# # Save to file
# os.makedirs("data", exist_ok=True)
# with open("data/trajectories_mf.json", "w") as f:
#     json.dump(trajectories_json.to_dict(orient="records"), f, indent=2)
# pymeos_finalize() 


# ::::::::::::::::::::::::::::::stopped here , time limit
# Storing in MobilityDB: so they use PYMEOS but here use server.py- susceptible d'etre modifié!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# https://stackoverflow.com/questions/51699817/python-async-post-requests


In [6]:

# def clean_up_resources():
 
#     try:
#         pymeos_finalize() 
#         print("MEOS resources cleaned up successfully.")
#     except Exception as e:
#         print(f"Error during resource cleanup: {str(e)}")

# server_url = 'http://localhost:8080/collections/{collectionId}/items'

# collection_id = 'ships'


# for index, row in trajectories.iterrows():
#     try:
    
#         post_data = {
#             "type":"Feature",
#             "id": row.name, 
#             "temporalGeometry":json.loads(row["trajectory"].as_mfjson())  
#         }

        
#         json_data = json.dumps(post_data)
# # post 
#         response = requests.post(
#             server_url.format(collectionId=collection_id), 
#             data=json_data, 
#             headers={'Content-Type': 'application/geo+json'}
#         )

     
#         if response.status_code == 200:
#             print(f"Successfully posted feature {row.name}")
#         else:
#             print(f"Failed to post feature {row.name}: {response.status_code} - {response.text}")
    
#     except requests.exceptions.RequestException as e:

#         print(f"Request error for feature {row.name}: {str(e)}")
    
#     except Exception as e:

#         print(f"Unexpected error for feature {row.name}: {str(e)}")

# clean_up_resources()
